# E1 — 1D double well: LSC-CP vs six baselines

Boltzmann target $\pi(x) \propto e^{-\beta V(x)}$ at $\beta = 8$ ($\varepsilon = 1/\beta = 0.125$) for the scaled quartic double well
$$V(x) = (x^2-1)^2 .$$
Seven samplers (ULA, MALA, FLA, kinetic Langevin/BAOAB, parallel tempering, raw compound-Poisson, LSC-CP) share one drift-taming policy, one $\Delta t$, one metric cadence, and per-seed initial conditions. LSC-CP adds nonlocal well-to-well jumps *plus* the stationary Lévy-score drift correction that preserves $\pi$ exactly at the generator level; raw CP has the same jumps without the correction. The wall-clock axis is a reported result: timing covers sampler work only, with CUDA synchronisation around every timed region.

In [1]:
EXPERIMENT = "double_well"
import os, sys, math, time, json
sys.path.insert(0, os.path.abspath(".."))
from src.gpu_guard import select_gpu
select_gpu(int(os.environ.get("JCP_GPU", "4")))
import torch
assert torch.cuda.device_count() == 1, "GPU guard must mask to exactly one device"
torch.set_default_dtype(torch.float64)
import numpy as np
import pandas as pd

from src import config as C
from src.experiments import build_e1, make_sampler_factory, make_metrics
from src.runner import (run_experiment, run_one, refine_dt, quadrature_refinement,
                        write_timeseries_csv, write_summary_csv, write_manifest,
                        ula_first_passage, hardware_manifest)
from src.samplers import tune_ladder
from src.certificate import make_phi_family, certificate_grid, certificate_importance
from src.plotting import make_all_figures, apply_style

DEV = "cuda"
RESULTS = os.path.abspath(os.path.join("..", "results", EXPERIMENT))
FIGURES = os.path.abspath(os.path.join("..", "figures", EXPERIMENT))
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
exp = build_e1(device=DEV)
cfg = exp.cfg
print(f"experiment={cfg.name}  d={cfg.d}  N={cfg.n_particles}  T={cfg.T}  dt0={cfg.dt}")
print(f"beta={cfg.beta}  eps={cfg.eps}  lambda={cfg.lam}  seeds={cfg.seeds}")
print(hardware_manifest())

experiment=double_well  d=1  N=4000  T=100.0  dt0=0.005
beta=8.0  eps=0.125  lambda=1.0  seeds=(0, 1, 2, 3, 4)


{'cpu': 'AMD EPYC 9554 64-Core Processor', 'gpu': 'NVIDIA H200 NVL', 'torch': '2.13.0+cu130', 'cuda': '13.0', 'python': '3.12.13', 'git_sha': 'b84ad342ffc64e1d0a657de15c492269bca0e65c', 'cuda_visible_devices': '5', 'gpu_compute_apps_at_start': ['GPU-7eefb508-cc10-0308-89bc-5e8ec2d31a7b, 1889891, 1240 MiB', 'GPU-c8768979-ae7d-8ca1-a33d-2874c9d21fae, 1694731, 28790 MiB', 'GPU-e003b5e1-3cce-0c27-5a6b-9f594c1b7682, 1694732, 29184 MiB', 'GPU-e003b5e1-3cce-0c27-5a6b-9f594c1b7682, 1933916, 632 MiB', 'GPU-0693e45d-5be4-82d1-e0fa-e72261e24e3b, 1597887, 108278 MiB', 'GPU-0693e45d-5be4-82d1-e0fa-e72261e24e3b, 1766636, 814 MiB', 'GPU-0693e45d-5be4-82d1-e0fa-e72261e24e3b, 1859671, 1752 MiB', 'GPU-b7477750-5992-8ae3-fde2-11bf70934062, 1694733, 29174 MiB', 'GPU-b7477750-5992-8ae3-fde2-11bf70934062, 1727069, 814 MiB', 'GPU-b7477750-5992-8ae3-fde2-11bf70934062, 1885971, 3426 MiB', 'GPU-b7477750-5992-8ae3-fde2-11bf70934062, 1930491, 7572 MiB']}


## The model

$$V(x) = (x^2-1)^2,\qquad V'(x) = 4x(x^2-1),$$
minima $\pm1$, saddle $0$, $V''(\pm1)=8$, $V''(0)=-4$, $\Delta V = 1$, so $\beta\Delta V = 8$. Kramers:
$$\tau = \frac{2\pi}{\sqrt{V''(\pm1)\,|V''(0)|}}\,e^{\beta\Delta V} = \frac{2\pi}{\sqrt{32}}e^{8} \approx 3.3\times10^3,$$
so over $T=100$ a ULA particle crosses with probability $\approx 3\%$. *(This is the standard $\tfrac{x^4}4 - \tfrac{x^2}2$ well scaled by 4; the unscaled version gives $\beta\Delta V = 2$, $\tau \approx 33$, and ULA would visibly cross.)*

Protocol: $N=4000$, $T=100$, $\Delta t_0 = 0.005$, box $[-3,3]$, $x_0 \sim \mathcal N(-1, 0.05^2)$. Partition ($K=2$): $\mathrm{sign}(x)$, $p^\star=(\tfrac12,\tfrac12)$ exactly by symmetry; additionally a 200-bin **density TV** against the exact $\pi$ on $[-3,3]$ (a genuine density TV, unlike the occupancy TV). Reference: inverse-CDF on a dense grid (tail mass outside $[-3,3]$ is $<10^{-30}$).

Below: barrier verification — ULA's empirical mean first-passage time out of the initial basin (censored-exponential MLE) against the Kramers estimate. Do not trust Kramers alone.

In [2]:
# critical points and curvatures, asserted
V = lambda x: (x**2 - 1.0)**2
dV = lambda x: 4.0*x*(x**2 - 1.0)
assert abs(V(1.0)) < 1e-15 and abs(V(-1.0)) < 1e-15 and V(0.0) == 1.0
assert dV(1.0) == 0.0 and dV(-1.0) == 0.0 and dV(0.0) == 0.0
d2V = lambda x: 12.0*x**2 - 4.0
assert d2V(1.0) == 8.0 and d2V(0.0) == -4.0
beta_dV = C.BETA * 1.0
print(f"beta*DeltaV = {beta_dV}, Kramers tau = {exp.kramers_tau:.1f}")

# barrier verification: ULA MFPT out of the left well vs Kramers
g = torch.Generator(device=DEV); g.manual_seed(0)
barrier_report = ula_first_passage(exp.pot, exp.box, exp.init_fn(cfg.n_particles, g),
                                   exp.in_basin0, cfg.dt, int(cfg.T/cfg.dt),
                                   C.EPS, g)
barrier_report["kramers_tau"] = exp.kramers_tau
print("ULA first-passage:", barrier_report)
print(f"measured MFPT {barrier_report['mfpt_estimate']:.0f} vs Kramers {exp.kramers_tau:.0f} "
      f"({barrier_report['n_exits']} exits of {barrier_report['n_particles']})")

beta*DeltaV = 8.0, Kramers tau = 3311.0


ULA first-passage: {'n_particles': 4000, 'T': 100.0, 'n_exits': 195, 'exit_fraction': 0.04875, 'mfpt_estimate': 2000.7275641025642, 'kramers_tau': 3311.0118449986508}
measured MFPT 2001 vs Kramers 3311 (195 exits of 4000)


## Jump law and Lévy score (generic shell)

Two-atom symmetric shell: centres $r_a = \pm 2$ with $w = (\tfrac12,\tfrac12)$, half-thickness $h=0.2$, $\lambda = 1$ — a jump of $\pm2$ maps minimum to minimum exactly, and the shell thickening $r = r_a + \rho u_a$, $\rho\sim\mathrm{Unif}(-h,h)$ avoids a purely atomic law.

The score uses Gauss–Legendre probability weights on both inner integrals ($Q_\theta$ nodes on $[0,1]$; $Q_\rho$ nodes on $[-h,h]$ matching $\rho\sim\mathrm{Unif}$), so the quadrature measure is exactly the sampler's $\nu$ — an invariant asserted in `tests/test_samplers.py`. **Log-space accumulation** (no score clipping): the per-direction integral $I_{a,q}$ is strictly positive but spans hundreds of orders of magnitude at $\beta=8$, so we form $\log I_{a,q}$ by log-sum-exp, extract $M(x) = \max_{a,q}[\log w_a + \log\hat w_q + \log I_{a,q}]$, build the $O(1)$ direction vector $v(x)$, and return $S = -\lambda\,e^{\min(M, 600)}v$. Because the drift is tamed, when $\|S\|$ is astronomical only its direction matters, and $v$ preserves the direction exactly. The fraction of particles hitting $M_{\max}$ (`m_clip_fraction`) and the running $\max M$ are logged.

Defaults $Q_\theta = 16$, $Q_\rho = 8$; the §9.5 quadrature-refinement table (certificate residual + terminal LSC-CP metrics per setting) is recorded in the run section below, and production uses the smallest setting converged against the finest.

In [3]:
print("atoms:", exp.law.atoms.squeeze(-1).tolist(), " weights:", exp.law.weights.tolist(),
      " h:", exp.law.h.tolist())
DEFAULT_QUAD = dict(q_theta=C.Q_THETA, q_rho=C.Q_RHO)
phis = make_phi_family(1, [0.0], 1.0, DEV)

def cert_e1(q_theta, q_rho, lo=-5.2, hi=5.2):
    score = exp.make_score(q_theta=q_theta, q_rho=q_rho)
    # jump side: FINE rho quadrature representing the continuous nu, so the
    # residual also sees any inadequacy of the score's rho order
    shifts, logw = exp.law.quadrature_shifts(64)
    return certificate_grid(exp.pot, score, shifts, logw, cfg.lam, cfg.beta,
                            phis, [lo], [hi], n_panels=120, nodes_per_panel=8)

atoms: [2.0, -2.0]  weights: [0.5, 0.5]  h: [0.2, 0.2]


## Target preservation: the stationarity identity $(\star)$

The LSC-CP generator is
$$\mathcal A f = \big[-\nabla V + S_{\nu,\beta}\big]\cdot\nabla f + \varepsilon\,\Delta f + \lambda\!\int\!\big[f(x+r)-f(x)\big]\nu(dr),$$
with $\nu$ a **probability** measure and
$$S_{\nu,\beta}(x) = -\lambda \int \nu(dr)\; r \int_0^1 \exp\!\Big[-\beta\big(V(x-\theta r) - V(x)\big)\Big]\, d\theta .$$
Raw CP is the same generator with $S \equiv 0$.

Write $p = e^{-\beta V}/Z$. The overdamped part is $\pi$-reversible, so invariance of $\pi$ is equivalent to
$$\int S\cdot\nabla\varphi \, d\pi + \int J\varphi \, d\pi = 0 \qquad \forall\, \varphi \in C_c^\infty . \tag{$\star$}$$

**Jump term.** Shift the integration variable and apply the fundamental theorem of calculus along $\theta \mapsto y - \theta r$:
$$\int J\varphi\,d\pi = \lambda\!\int\!\nu(dr)\!\int\!\varphi(y)\big[p(y-r)-p(y)\big]dy = -\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\varphi(y)\,r\!\cdot\!\nabla p(y-\theta r)\,dy\,d\theta.$$

**Drift term.** $S(x) = -\lambda\int\nu(dr)\,r\int_0^1 \frac{p(x-\theta r)}{p(x)}d\theta$ (identical to the boxed formula since $p \propto e^{-\beta V}$), so integrating by parts in $x$:
$$\int S\cdot\nabla\varphi\,d\pi = -\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\big(r\!\cdot\!\nabla\varphi(x)\big) p(x-\theta r)\,dx\,d\theta = +\lambda\!\int\!\nu(dr)\!\int_0^1\!\!\int\!\varphi(x)\,r\!\cdot\!\nabla p(x-\theta r)\,dx\,d\theta.$$

The two cancel identically. **Target preservation is unconditional in $\nu$** — any finite-activity jump law works; only the *speed* depends on $\nu$.

### The measured certificate $\mathcal R(\varphi)$

For smooth bounded test functions (products of tanh ridges) we report
$$\mathcal R(\varphi) = \frac{\big|\int S_{\nu,\beta}\!\cdot\!\nabla\varphi\,d\pi + \int J_\nu\varphi\,d\pi\big|}{\big|\int J_\nu\varphi\,d\pi\big|},$$
zero in exact arithmetic; the measured value is the combined defect of the $\theta$/$\rho$ quadratures. Two implementation notes, both load-bearing:

1. **The integration domain extends a full jump length beyond the target's effective support.** Order-one contributions to $(\star)$ live where $\pi$ is tiny and $S$ is enormous; a deliberately tight box produces a large residual (demonstrated below and regression-tested).
2. **The drift integrand $p\,S\cdot\nabla\varphi$ is assembled in log space** from the score's $(M, v)$ parts as $\exp(-\beta V + M)\,v\cdot\nabla\varphi$: in linear fp64 arithmetic $p$ underflows exactly where $\|S\|$ is astronomical, silently dropping those order-one far-field contributions. The residual uses the *uncapped* $M$; the deployed drift caps $M$ at $M_{\max}=600$, but because taming saturates (the tamed step tends to $-v/\|v\|$), the deployed tamed step differs from the uncapped one by $O(e^{-M_{\max}})$ — that saturation defect is reported alongside $\mathcal R$ and is $\lesssim 10^{-250}$ here.

A useful exact identity (change of variables $x \to x+\theta_p r$ in the drift term): for the *implemented* quadrature score,
$$\int S\cdot\nabla\varphi\,d\pi + \int J\varphi\,d\pi \;=\; \lambda\,\mathbb E_\pi\!\int\!\nu(dr)\Big[\varphi(x+r)-\varphi(x) - \sum_p \hat w_p\, r\cdot\nabla\varphi(x+\theta_p r)\Big],$$
i.e. the residual is **independent of $V$** and equals the $\theta$-quadrature error on the smooth test-function integrand. This is why moderate pointwise errors of the Gauss–Legendre rule on the stiff factor $e^{\beta\Delta V}$ do not translate into a weak (distributional) defect of the sampled law.

In [4]:
cert_report = cert_e1(**DEFAULT_QUAD)
print(f"R(phi) over the generous box [-5.2, 5.2] (support [-3,3] + jump reach 2.2):")
for i in range(len(phis)):
    print(f"  phi_{i}: R = {cert_report[f'phi_{i}']['residual']:.3e}"
          f"  (jump term {cert_report[f'phi_{i}']['jump_term']:+.3e})")
print(f"max R = {cert_report['max_residual']:.3e}  "
      f"clip saturation defect = {cert_report['clip_tamed_step_defect']:.2e}")
assert cert_report["max_residual"] < 1e-6

tight = cert_e1(**DEFAULT_QUAD, lo=-1.3, hi=1.3)
print(f"DELIBERATELY TIGHT box [-1.3, 1.3]: max R = {tight['max_residual']:.3e} "
      "-- order-one contributions to the identity live beyond the target support")

R(phi) over the generous box [-5.2, 5.2] (support [-3,3] + jump reach 2.2):
  phi_0: R = 1.778e-13  (jump term +1.954e-02)
  phi_1: R = 1.291e-10  (jump term +8.866e-02)
  phi_2: R = 1.543e-12  (jump term -1.283e-02)
  phi_3: R = 3.015e-12  (jump term +1.013e-01)
  phi_4: R = 5.474e-08  (jump term -2.872e-03)
  phi_5: R = 3.719e-09  (jump term +5.286e-02)
max R = 5.474e-08  clip saturation defect = 1.21e-261
DELIBERATELY TIGHT box [-1.3, 1.3]: max R = 4.233e-01 -- order-one contributions to the identity live beyond the target support


## The seven methods

All methods share one **taming policy**: the same map $b \mapsto b/(1+\Delta t\,\|b\|)$ is applied to every method's drift (ULA, the MALA proposal, FLA, the BAOAB force, raw CP, LSC-CP). Tamed MALA is still exact because the proposal density $q(y|x) = \mathcal N(y;\, x + \Delta t\, b_{\rm tamed}(x),\, 2\varepsilon\Delta t\, I)$ is used consistently in both directions of the MH ratio; asymmetric taming would make taming a hidden variable in the comparison.

**1. ULA.** $X \leftarrow X + \Delta t\,\mathrm{tame}(-\nabla V) + \sqrt{2\varepsilon\Delta t}\,\xi$.

**2. MALA.** With $\nabla\log\pi = -\beta\nabla V$, the proposal $Y = X - \tfrac{h\beta}2\nabla V(X) + \sqrt h\,\xi$ matches the ULA step iff $\tfrac{h\beta}{2} = \Delta t$ **and** $h = 2\varepsilon\Delta t$. Both conditions coincide:
$$\boxed{h = \frac{2\Delta t}{\beta} = 2\varepsilon\Delta t = \Delta t/4 \quad\text{at }\beta=8.}$$
Log-acceptance
$$\log\alpha = -\beta[V(Y)-V(X)] - \frac{1}{2h}\Big[\|X - \mu(Y)\|^2 - \|Y-\mu(X)\|^2\Big],\qquad \mu(z) = z + \Delta t\,\mathrm{tame}(-\nabla V(z)),$$
accepted elementwise. Proposals are **never clipped before the accept step** (that silently breaks exactness); out-of-box proposals are auto-rejected, which is valid MH for the box-restricted target. Expect acceptance $\approx 1$ — the honest message: **rejection does not cure metastability**.

**3. FLA / FLMC** (Şimşekli, ICML 2017, §3.3). $X \leftarrow X + \Delta t\,\mathrm{tame}(-c_\alpha \nabla U) + \Delta t^{1/\alpha}\,\xi^{(\alpha)}$ with $U = \beta V$, $c_\alpha = \Gamma(\alpha-1)/\Gamma(\alpha/2)^2$, $\alpha = 1.7$; per-coordinate $S\alpha S(1)$ noise by Chambers–Mallows–Stuck. **No tail clipping** — a truncated stable is not stable. FLA is the *uncorrected nonlocal* comparator: heavy tails cross barriers, but the invariant law is not $\pi$.

**4. Kinetic Langevin (BAOAB).** It is **not HMC** (no accept/reject; carries $O(\Delta t^2)$ configurational bias). Unit mass, $\gamma = 1$; the O-step coefficient is the exact OU solution: $dp = -\gamma p\,dt + \sqrt{2\gamma\varepsilon}\,dW$ gives $\mathrm{Var} = 2\gamma\varepsilon\int_0^{\Delta t}e^{-2\gamma s}ds = \varepsilon(1-e^{-2\gamma\Delta t})$ by Itô isometry. The trailing force is cached as the next step's leading B (one gradient per step).

**5. Parallel tempering.** MALA-within-replica at $\beta_k = \beta\, r^{k-1}$, replica $k$ using $h_k = 2\Delta t/\beta_k$ (same tamed drift step for every replica; only the noise scale differs). Adjacent swaps every $n_{\rm swap}$ steps (alternating parity); the joint target is $\prod_k \pi_k$ and the swap is a deterministic involution, so
$$\alpha_{\rm swap} = \min\Big\{1,\ \exp\big[(\beta_i - \beta_{i+1})\big(V(x_i) - V(x_{i+1})\big)\big]\Big\}.$$
$V$ values are cached by MALA, so swaps are free in evaluation count. $K$ is tuned so the mean swap acceptance lands in $[0.2, 0.4]$. The replica index is a batch dimension, state $(K, N, d)$; metrics use the cold replica only; **wall-clock includes all $K$ replicas**.

**6/7. Raw CP and LSC-CP.** Identical time discretisation,
$$X^{(1)} = X_n + \Delta t\,\frac{b(X_n)}{1+\Delta t\,\|b(X_n)\|} + \sqrt{2\varepsilon\Delta t}\;\xi_n,\qquad X_{n+1} = X^{(1)} + \sum_{k=1}^{N_n} A_k,$$
$N_n \sim \mathrm{Poisson}(\lambda\Delta t)$, $A_k \stackrel{iid}\sim \nu$; $b = -\nabla V$ (raw CP) or $b = -\nabla V + S_{\nu,\beta}$ (LSC-CP). The jump stream is a dedicated generator seeded identically for both methods, so their jump times and increments are **pathwise identical** (verified in `tests/test_samplers.py`), not merely equal in law.

In [5]:
# PT ladder: geometric in beta, K tuned so mean swap acceptance is in [0.2, 0.4]
gen = torch.Generator(device=DEV); gen.manual_seed(0)
x0_pilot = exp.init_fn(min(512, cfg.n_particles), gen)
pt_betas, ladder_info = tune_ladder(exp.pot, x0_pilot, cfg.dt, exp.box,
                                    C.BETA, exp.pt_beta_min, pilot_steps=600)
print(f"PT ladder: K={ladder_info['K']}  r={ladder_info['r']:.4f}  "
      f"beta_K={pt_betas[-1].item():.4f}  swap acceptance={ladder_info['swap_acceptance']:.3f}")
print("tuning history {K: acceptance}:", ladder_info["history"])

PT ladder: K=2  r=0.1250  beta_K=1.0000  swap acceptance=0.418
tuning history {K: acceptance}: {8: 0.9050699869791666, 6: 0.8648111979166669, 4: 0.7769694010416667, 3: 0.6782877604166666, 2: 0.4176432291666667}


## Reference, partition, metrics, bias floors

Reference sample size equals the run's $N$; metrics are evaluated at every checkpoint (cadence fixed in $t$, identical across methods).

* **$W_2$**: exact in 1D (sorted coupling); **sliced** $W_2$ for $d\ge2$ with $L=200$ projections drawn once from a fixed seed and reused across all times and methods (its bias floor decays like $N^{-1/2}$, not $N^{-1/d}$).
* **TV** (occupancy, on the partition): $\tfrac12\sum_k|\hat p_k - p^\star_k|$ — a **lower bound** on the full TV.
* **MMD**: Gaussian kernel, bandwidth **frozen once** by the median heuristic on the reference sample (per-frame bandwidths would make curves non-comparable); biased V-statistic $\widehat{\mathrm{MMD}}_b^2 = \|\mu_X-\mu_Y\|_{\mathcal H}^2 \ge 0$.
* **EMC** $= e^{H(\hat p)}/K$: plotted with a horizontal line at the target $e^{H(p^\star)}/K$; EMC $=1$ is optimal only for uniform $p^\star$, and deviation in *either* direction is error.
* **EJS**: base-2 Jensen–Shannon divergence between $\hat p$ and $p^\star$ (Blessing et al., arXiv:2406.07423, App. A.3), bounded in $[0,1]$, quadratic near the target, so it stays informative where TV saturates.
* **Bias floors** (mandatory): each metric between two independent reference samples of size $N$, 20 replicates; dashed line on every panel. Without this, every plateau is uninterpretable.
* **Nonfinite fraction**: logged per method per checkpoint; must be identically zero — metrics on survivors only would be survivorship bias, so nothing is ever filtered.

**Coverage vs correctness, once:** EMC measures *coverage*, TV/EJS measure *correctness*. Raw CP, whose invariant law is not $\pi$, over-flattens — driving EMC toward 1 (above its target line) while TV and EJS stay bad. That pairing *is* the raw-CP-vs-LSC-CP story.

In [6]:
metrics_fn, floors, aux = make_metrics(exp, cfg.n_particles)
emc_target = exp.emc_target
print("p_star:", np.round(exp.p_star.cpu().numpy(), 6))
print("EMC target line: %.4f" % emc_target)
print("MMD bandwidth (median heuristic on reference, frozen):", round(aux["bandwidth"], 4))
print("bias floors (mean +- std over 20 replicate pairs):")
for k, v in floors.items():
    print(f"  {k:>12s}: {v['mean']:.5f} +- {v['std']:.5f}")

p_star: [0.5 0.5]
EMC target line: 1.0000
MMD bandwidth (median heuristic on reference, frozen): 1.0684
bias floors (mean +- std over 20 replicate pairs):
            W2: 0.08802 +- 0.06279
           MMD: 0.01054 +- 0.01063
            TV: 0.00483 +- 0.00533
           EJS: 0.00004 +- 0.00008
           EMC: 0.99990 +- 0.00022
    TV_density: 0.04076 +- 0.00483


## $\Delta t$ refinement and production

Declared $\Delta t$ selection rule, applied uniformly to every experiment (reported in the SI): **the largest $\Delta t$ on a dyadic grid at which every method's terminal value of every metric is within 5% of its $\Delta t/2$ value.** Three statistical guards make the rule meaningful at a single refinement seed: differences are measured relative to $\max(|m_{\Delta t/2}|,\ \text{bias floor})$; when *both* values sit inside the floor band (floor mean $+\,3$ s.d.) they are declared in agreement; and differences within $4\times$ the floor s.d. — the natural unit of single-run metric sampling noise at this $N$ — are likewise noise, not discretisation bias. The same guards apply to the quadrature-refinement comparison.

**One declared exception:** FLA does not gate the $\Delta t$ selection. Its continuum limit is not $\pi$, so its bias has no $\Delta t$ at which it should stabilise (empirically its density error *drifts monotonically* under refinement); demanding 5% stability from it would refine $\Delta t$ forever. FLA still runs at the shared chosen $\Delta t$, and its deviations across the dyadic grid are recorded in the refinement table for transparency.

Production protocol: 5 seeds $\times$ 7 methods, run **sequentially** (never batched) so per-run wall-clock is meaningful; all methods share $x_0$ per seed; 20 untimed warm-up steps absorb allocator/JIT effects; `torch.cuda.synchronize()` brackets every timed region, and the timer covers sampler work only.

In [7]:
def run_terminal_lsc(**quad):
    f = make_sampler_factory(exp, cfg.dt, pt_betas, score_kwargs=quad)
    n_ = int(round(cfg.T / cfg.dt))
    r_, _ = run_one("LSC-CP", 0, f, n_, n_, cfg.dt, metrics_fn, exp.pot, quiet=True)
    return {k: r_[-1][k] for k in ("W2", "TV", "TV_density", "MMD", "EMC", "EJS")}

settings = [dict(q_theta=qt, q_rho=qr) for qt in (8, 16, 32) for qr in (4, 8, 16)]
CHOSEN_QUAD, quad_table = quadrature_refinement(
    settings, run_terminal_lsc, lambda **s: cert_e1(**s)["max_residual"], floors)
print("chosen production quadrature:", CHOSEN_QUAD)
display(pd.DataFrame(quad_table).round(6))
if CHOSEN_QUAD != DEFAULT_QUAD:
    cert_report = cert_e1(**CHOSEN_QUAD)
    print("certificate re-evaluated at chosen orders: max R =",
          f"{cert_report['max_residual']:.3e}")
    assert cert_report["max_residual"] < 1e-6

chosen production quadrature: {'q_theta': 16, 'q_rho': 4}


,q_theta,q_rho,R,W2,TV,TV_density,MMD,EMC,EJS,pass
0,8,4,0.006608,0.103826,0.00275,0.066262,0.018471,0.999985,0.000005,False
1,8,8,0.006608,0.103826,0.00275,0.066262,0.018470,0.999985,0.000005,False
2,8,16,0.006608,0.103826,0.00275,0.066262,0.018470,0.999985,0.000005,False
3,16,4,0.000000,0.107370,0.00350,0.062075,0.016134,0.999976,0.000009,True
4,16,8,0.000000,0.107369,0.00350,0.062075,0.016134,0.999976,0.000009,True
5,16,16,0.000000,0.107369,0.00350,0.062075,0.016134,0.999976,0.000009,True
6,32,4,0.000000,0.107434,0.00350,0.062109,0.016135,0.999976,0.000009,True
7,32,8,0.000000,0.107434,0.00350,0.062109,0.016135,0.999976,0.000009,True
8,32,16,0.000000,0.107434,0.00350,0.062109,0.016135,0.999976,0.000009,True


certificate re-evaluated at chosen orders: max R = 1.483e-07


In [8]:
MAIN_METRICS = ["W2", "TV", "TV_density", "MMD", "EMC", "EJS"]

def run_terminal_all(dt_):
    n_ = int(round(cfg.T / dt_))
    factory = make_sampler_factory(exp, dt_, pt_betas, score_kwargs=CHOSEN_QUAD)
    out = {}
    for m in C.METHODS:
        rows_, _ = run_one(m, 0, factory, n_, n_, dt_, metrics_fn, exp.pot, quiet=True)
        out[m] = {k: rows_[-1][k] for k in MAIN_METRICS}
    print(f"  refine_dt: finished pass at dt={dt_}", flush=True)
    return out

dt_final, dt_table = refine_dt(run_terminal_all, cfg.dt, floors, exclude=("FLA",))
print("chosen dt:", dt_final)
for row in dt_table:
    print(row)

n_steps = int(round(cfg.T / dt_final))
steps_per_ck = max(1, n_steps // C.N_CHECKPOINTS)
factory = make_sampler_factory(exp, dt_final, pt_betas, score_kwargs=CHOSEN_QUAD)
t0 = time.time()
rows, method_info = run_experiment(C.METHODS, cfg.seeds, factory, n_steps,
                                   steps_per_ck, dt_final, metrics_fn, exp.pot)
print(f"production total: {time.time()-t0:.0f}s")
worst_nonfinite = max(r["nonfinite_frac"] for r in rows)
assert worst_nonfinite == 0.0, worst_nonfinite
print("nonfinite fraction: identically zero across all methods/checkpoints")

  refine_dt: finished pass at dt=0.005


  refine_dt: finished pass at dt=0.0025


chosen dt: 0.005
{'dt': 0.005, 'pass': True, 'failures': [], 'excluded_deviations': []}


  ULA seed 0: 2.8s sampler wall-clock


  ULA seed 1: 2.8s sampler wall-clock


...

  LSC-CP seed 2: 13.4s sampler wall-clock


  LSC-CP seed 3: 13.4s sampler wall-clock


  LSC-CP seed 4: 13.4s sampler wall-clock


LSC-CP: done in 68.0s total


production total: 306s
nonfinite fraction: identically zero across all methods/checkpoints


## Figures

One figure per metric, all seven methods, versus $t=n\Delta t$ and versus wall-clock; mean over 5 seeds with a pre-blended $\pm1$ s.d. band (EPS forbids transparency). Dashed lines: bias floors (log-scale metrics) and the EMC target. Saved individually in `.pdf`, `.png` (600 dpi) and `.eps` with caption `.txt` files.

In [9]:
fig_metrics = ("W2", "TV", "TV_density", "MMD", "EMC", "EJS")
written = make_all_figures(rows, FIGURES, floors, emc_target, metrics=fig_metrics)
print(f"{len(written)} figures x 3 formats (.pdf/.png 600dpi/.eps) + captions -> {FIGURES}")

# grid display for inspection (saved files above are one-figure-per-file)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
n_m = len(fig_metrics)
fig, axes = plt.subplots(n_m, 2, figsize=(11, 3.2 * n_m))
for i, metric in enumerate(fig_metrics):
    for j, tag in enumerate(("vs_time", "vs_wallclock")):
        ax = axes[i, j] if n_m > 1 else axes[j]
        ax.imshow(mpimg.imread(os.path.join(FIGURES, f"{metric}_{tag}.png")))
        ax.set_axis_off()
plt.tight_layout(); plt.show()

12 figures x 3 formats (.pdf/.png 600dpi/.eps) + captions -> /home/zheyuanlai/levy-sampling/JCP_experiments/figures/double_well


## CSV emission and summary

In [10]:
ts_path = os.path.join(RESULTS, "metrics_timeseries.csv")
write_timeseries_csv(rows, ts_path)
summary_metrics = MAIN_METRICS + ["nonfinite_frac"]
summary = write_summary_csv(rows, C.METHODS, cfg.seeds, summary_metrics,
                            method_info, floors, os.path.join(RESULTS, "summary.csv"))

manifest = dict(
    experiment=EXPERIMENT,
    config=dict(d=cfg.d, N=cfg.n_particles, T=cfg.T, dt0=cfg.dt, dt=dt_final,
                beta=cfg.beta, eps=cfg.eps, lam=cfg.lam, seeds=list(cfg.seeds),
                n_checkpoints=C.N_CHECKPOINTS, warmup_steps=C.N_WARMUP_STEPS),
    quadrature=dict(chosen=CHOSEN_QUAD, table=quad_table),
    dt_refinement=[{k: (str(v) if isinstance(v, tuple) else v) for k, v in row.items()}
                   for row in dt_table],
    pt_ladder={k: v for k, v in ladder_info.items()},
    certificate=cert_report,
    bias_floors=floors,
    barrier_verification=barrier_report,
    method_info={m: {k: v for k, v in mi.items() if isinstance(v, (int, float))}
                 for m, mi in method_info.items()},
    hardware=hardware_manifest(),
    
)
write_manifest(os.path.join(RESULTS, "manifest.json"), **manifest)
print("wrote", ts_path)
from IPython.display import display
display(pd.read_csv(os.path.join(RESULTS, "summary.csv")).round(5))

wrote /home/zheyuanlai/levy-sampling/JCP_experiments/results/double_well/metrics_timeseries.csv


,method,EJS_mean,EJS_std,EMC_mean,EMC_std,MMD_mean,MMD_std,TV_density_mean,TV_density_std,TV_mean,...,jump_count_mean,m_clip_fraction,mala_accept,nonfinite_frac_mean,nonfinite_frac_std,pt_swap_accept,score_quad_evals_per_step,time_to_threshold_TV,wallclock_mean_s,wallclock_std_s
0,ULA,0.24048,0.00492,0.56828,0.00539,0.58987,0.00359,0.47252,0.00258,0.47190,...,NaN,NaN,NaN,0.0,0.0,NaN,0.0,inf,2.77110,0.04096
1,MALA,0.24272,0.00230,0.56578,0.00254,0.59224,0.00190,0.47370,0.00132,0.47315,...,NaN,NaN,0.99763,0.0,0.0,NaN,0.0,inf,8.20928,0.00797
2,FLA,0.00000,0.00000,0.99999,0.00001,0.00355,0.00136,0.13174,0.00854,0.00145,...,NaN,NaN,NaN,0.0,0.0,NaN,0.0,14.0,4.50580,0.02579
3,BAOAB,0.27697,0.00221,0.53048,0.00213,0.61273,0.00090,0.49003,0.00093,0.48930,...,NaN,NaN,NaN,0.0,0.0,NaN,0.0,inf,3.49931,0.00147
4,PT,0.00002,0.00003,0.99995,0.00008,0.00623,0.00334,0.04500,0.00512,0.00390,...,NaN,NaN,0.99722,0.0,0.0,0.40504,0.0,22.0,9.00149,0.02839
5,CP,0.00002,0.00002,0.99995,0.00007,0.03177,0.00292,0.09729,0.00332,0.00430,...,0.00503,NaN,NaN,0.0,0.0,NaN,0.0,4.0,7.91065,0.03268
6,LSC-CP,0.00002,0.00002,0.99994,0.00006,0.01610,0.00073,0.06484,0.00294,0.00470,...,0.00503,0.0,NaN,0.0,0.0,NaN,512000.0,4.0,13.45044,0.00752
